In [ ]:

import os
import cv2
import numpy as np
import mediapipe as mp
import tensorflow as tf

MAX_FRAMES = 110

# Ordered exactly like the training label construction:
#   0=Jab, 1=Cross, 2=Lead Hook, 3=Rear Hook,
#   4=Lead Uppercut, 5=Rear Uppercut, 6=NoPunch
# (i.e. the same order as x_jab_aug + x_cross_aug + x_leadhook_aug +
#  x_rearhook_aug + x_leadUppercut_aug + x_rearUppercut_aug + x_nopunch_aug
#  in your training notebook.)
#
# Flip a class's "enabled" flag to False when you're testing a model
# variant that was trained WITHOUT that class (e.g. your 5-class runs).
# CLASS_NAMES below is built by filtering this list in order, so the
# resulting index->name mapping always lines up with a training run
# that comments out the SAME classes in the SAME order. If you add or
# remove a class here, make the matching change in the training
# notebook's data-loading/concatenation cells before you retrain --
# these two lists have to move together.

#Turn those classes as True which your model contains

ALL_CLASSES = [
    #("Jab",            True),
    #("Lead Hook",      True),
    #("cross",      True),
    #("Rear Uppercut",  True),
    #("cross_Rear Hook",      True),
    #("Lead Uppercut",  True),   
    #("Rear Uppercut",  True),   
    #("uppercuts_nopunch",        True),
]
CLASS_NAMES = [name for name, enabled in ALL_CLASSES if enabled]

MODEL_PATH = r"Add Model Path"
INPUT_DIR = r"Input Folder Path"
OUTPUT_DIR = r"Output Folder Path"

CONFIDENCE_THRESHOLD = 0.50 

# ==========================================
# 2. Joint mapping
#    Same 16 core joints, same order, as remove_z_coordinates +
#    remove_noisy_coordinates produced during training.
# ==========================================
MP_LANDMARK_INDICES = [11, 12, 13, 14, 15, 16, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32]
NUM_JOINTS = len(MP_LANDMARK_INDICES)  # 16

# Indices *within* the reduced 16-joint frame (0-15), matching the
# joint ids used by extract_angles / build_feature_vector during training.
LS, RS = 0, 1
LE, RE = 2, 3
LW, RW = 4, 5
LHIP, RHIP = 6, 7
LKNEE, RKNEE = 8, 9
LANKLE, RANKLE = 10, 11
LHEEL, RHEEL = 12, 13
LFOOT, RFOOT = 14, 15

BONES = [
    (LS, RS),
    (LS, LE), (LE, LW),
    (RS, RE), (RE, RW),
    (LS, LHIP), (RS, RHIP),
    (LHIP, RHIP),
    (LHIP, LKNEE), (LKNEE, LANKLE),
    (RHIP, RKNEE), (RKNEE, RANKLE),
    (LANKLE, LHEEL), (LHEEL, LFOOT),
    (RANKLE, RHEEL), (RHEEL, RFOOT),
]


# ==========================================
# 3. Raw extraction (single pass, no cleaning)
#    Mirrors the extraction script that built jab_data/cross_data/etc:
#    Pose(static_image_mode=False) with every other MediaPipe setting
#    left at its default (model_complexity=1, smooth_landmarks=True,
#    min_detection_confidence=0.5, min_tracking_confidence=0.5) --
#    just read x,y for the core joints every frame it's detected,
#    skip frames with no detection.
# ==========================================
def extract_raw_sequence(video_path, pose_detector):
    """Returns (feature_frames, pixel_frames, width, height, fps).

    feature_frames: list of 32-dim lists (x,y for 16 joints, 0-1
                     normalized MediaPipe coords) - used for classification.
    pixel_frames:   list of (16,2) arrays in pixel space - used only for
                     drawing the skeleton video.
    """
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        return [], [], None, None, None

    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = cap.get(cv2.CAP_PROP_FPS)
    if fps <= 0 or np.isnan(fps):
        fps = 30.0

    feature_frames = []
    pixel_frames = []

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        image_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        results = pose_detector.process(image_rgb)

        if not results.pose_landmarks:
            continue  # skip undetected frames, same as training extraction

        landmarks = results.pose_landmarks.landmark

        norm_frame = []
        pixel_frame = []
        for idx in MP_LANDMARK_INDICES:
            lm = landmarks[idx]
            norm_frame.extend([lm.x, lm.y])
            pixel_frame.append([lm.x * width, lm.y * height])

        feature_frames.append(norm_frame)
        pixel_frames.append(np.array(pixel_frame))

    cap.release()
    return feature_frames, pixel_frames, width, height, fps


# ==========================================
# 4. Normalization (identical to training: translation_normalization
#    then scale_normalization, in that order)
# ==========================================
def get_hip_center(frame):
    return np.array([(frame[LHIP * 2] + frame[RHIP * 2]) / 2.0,
                      (frame[LHIP * 2 + 1] + frame[RHIP * 2 + 1]) / 2.0])


def get_shoulder_center(frame):
    return np.array([(frame[LS * 2] + frame[RS * 2]) / 2.0,
                      (frame[LS * 2 + 1] + frame[RS * 2 + 1]) / 2.0])


def translation_normalization(frame):
    hip_center = get_hip_center(frame)
    normalized = []
    for i in range(0, len(frame), 2):
        joint_xy = np.array([frame[i], frame[i + 1]]) - hip_center
        normalized.extend(joint_xy.tolist())
    return normalized


def scale_normalization(frame):
    hip_center = get_hip_center(frame)
    shoulder_center = get_shoulder_center(frame)
    torso = np.linalg.norm(shoulder_center - hip_center)
    if torso < 1e-6:
        torso = 1e-6
    return [v / torso for v in frame]


def normalize_sequence(feature_frames):
    translated = [translation_normalization(f) for f in feature_frames]
    scaled = [scale_normalization(f) for f in translated]
    return scaled


# ==========================================
# 5. Feature engineering (identical to training)
# ==========================================
def calculate_angle(a, b, c):
    a, b, c = np.array(a), np.array(b), np.array(c)
    radians = np.arctan2(c[1] - b[1], c[0] - b[0]) - np.arctan2(a[1] - b[1], a[0] - b[0])
    angle = abs((180.0 * radians) / np.pi)
    return 360.0 - angle if angle > 180.0 else angle


def extract_angles(frame):
    def get_xy(idx):
        return [frame[idx * 2], frame[idx * 2 + 1]]

    return [
        calculate_angle(get_xy(LS), get_xy(LE), get_xy(LW)),
        calculate_angle(get_xy(RS), get_xy(RE), get_xy(RW)),
        calculate_angle(get_xy(LHIP), get_xy(LS), get_xy(LE)),
        calculate_angle(get_xy(RHIP), get_xy(RS), get_xy(RE)),
        calculate_angle(get_xy(LS), get_xy(LHIP), get_xy(LKNEE)),
        calculate_angle(get_xy(RS), get_xy(RHIP), get_xy(RKNEE)),
    ]


def extract_velocities(curr_frame, prev_frame):
    velocities = []
    for j_id in [LS, RS, LE, RE, LW, RW]:
        vx = curr_frame[j_id * 2] - prev_frame[j_id * 2]
        vy = curr_frame[j_id * 2 + 1] - prev_frame[j_id * 2 + 1]
        velocities.extend([vx, vy])
    return velocities


def extract_wrist_speeds(velocities):
    lw_speed = np.sqrt(velocities[8] ** 2 + velocities[9] ** 2)
    rw_speed = np.sqrt(velocities[10] ** 2 + velocities[11] ** 2)
    return [lw_speed, rw_speed]


def wrist_to_shoulder_distance(frame):
    def get_xy(idx):
        return np.array([frame[idx * 2], frame[idx * 2 + 1]])

    dist_l = np.linalg.norm(get_xy(LW) - get_xy(LS))
    dist_r = np.linalg.norm(get_xy(RW) - get_xy(RS))
    return [dist_l, dist_r]


def lead_rear_asymmetry(frame):
    return [frame[LW * 2] - frame[RW * 2]]


def wrist_motion_angle(velocities):
    """Direction of travel (atan2(vy, vx), in radians) for each wrist.
    velocities is ordered LS,RS,LE,RE,LW,RW (pairs of vx,vy) from
    extract_velocities, so LW sits at [8,9] and RW at [10,11]. Roughly
    0/pi (horizontal) for hooks, roughly +-pi/2 (vertical) for uppercuts."""
    lw_angle = np.arctan2(velocities[9], velocities[8])
    rw_angle = np.arctan2(velocities[11], velocities[10])
    return [lw_angle, rw_angle]


def cumulative_wrist_displacement(curr_frame, frame0):
    """Vertical and horizontal displacement of each wrist relative to the
    punch's first frame (frame0), not the previous frame. Returns
    (vert_disp, horiz_disp), each [left, right]. At frame 0 itself, this is
    called with frame0 == curr_frame, which naturally yields all zeros."""
    def get_xy(frame, idx):
        return frame[idx * 2], frame[idx * 2 + 1]

    lw_x0, lw_y0 = get_xy(frame0, LW)
    rw_x0, rw_y0 = get_xy(frame0, RW)
    lw_x, lw_y = get_xy(curr_frame, LW)
    rw_x, rw_y = get_xy(curr_frame, RW)

    vert_disp = [lw_y - lw_y0, rw_y - rw_y0]
    horiz_disp = [lw_x - lw_x0, rw_x - rw_x0]
    return vert_disp, horiz_disp


def vertical_horizontal_ratio(vert_disp, horiz_disp):
    """abs(vertical displacement) / abs(horizontal displacement) per wrist.
    High for uppercuts, low for hooks. +1e-6 guards divide-by-zero."""
    lw_ratio = abs(vert_disp[0]) / (abs(horiz_disp[0]) + 1e-6)
    rw_ratio = abs(vert_disp[1]) / (abs(horiz_disp[1]) + 1e-6)
    return [lw_ratio, rw_ratio]


def elbow_shoulder_height(frame):
    """elbow_y - shoulder_y for each arm. Computed directly on the current
    frame (not a delta) -- no frame-0 special case needed."""
    le_height = frame[LE * 2 + 1] - frame[LS * 2 + 1]
    re_height = frame[RE * 2 + 1] - frame[RS * 2 + 1]
    return [le_height, re_height]


def elbow_torso_offset(frame):
    """elbow_x - hip_center_x for each arm. Also a direct per-frame value,
    no frame-0 special case needed."""
    hip_center_x = (frame[LHIP * 2] + frame[RHIP * 2]) / 2.0
    le_offset = frame[LE * 2] - hip_center_x
    re_offset = frame[RE * 2] - hip_center_x
    return [le_offset, re_offset]
# ----------------------------------------------------------------------


def build_feature_vector(curr_frame, prev_frame, frame0):
    angles = extract_angles(curr_frame)
    velocities = extract_velocities(curr_frame, prev_frame)
    speeds = extract_wrist_speeds(velocities)
    wrist_dist = wrist_to_shoulder_distance(curr_frame)
    asymmetry = lead_rear_asymmetry(curr_frame)

    motion_angle = wrist_motion_angle(velocities)
    vert_disp, horiz_disp = cumulative_wrist_displacement(curr_frame, frame0)
    vh_ratio = vertical_horizontal_ratio(vert_disp, horiz_disp)
    elbow_height = elbow_shoulder_height(curr_frame)
    elbow_offset = elbow_torso_offset(curr_frame)

    return np.concatenate([
        curr_frame, velocities, angles, speeds, wrist_dist, asymmetry,
        motion_angle, vert_disp, horiz_disp, vh_ratio, elbow_height, elbow_offset
    ])


def process_punch_sequence(punch):
    if len(punch) == 0:
        return []

    features = []
    zero_vel = [0.0] * 12
    zero_speed = [0.0, 0.0]
    zero_asym = [0.0]

    frame0 = punch[0] 

    first_angles = extract_angles(punch[0])
    first_wrist_dist = wrist_to_shoulder_distance(punch[0])

    first_motion_angle = wrist_motion_angle(zero_vel)  
    first_vert_disp, first_horiz_disp = cumulative_wrist_displacement(punch[0], frame0)
    first_vh_ratio = vertical_horizontal_ratio(first_vert_disp, first_horiz_disp)
    first_elbow_height = elbow_shoulder_height(punch[0])
    first_elbow_offset = elbow_torso_offset(punch[0])

    features.append(np.concatenate([
        punch[0], zero_vel, first_angles, zero_speed, first_wrist_dist, zero_asym,
        first_motion_angle, first_vert_disp, first_horiz_disp, first_vh_ratio,
        first_elbow_height, first_elbow_offset
    ]))

    for i in range(1, len(punch)):
        features.append(build_feature_vector(punch[i], punch[i - 1], frame0))

    return features


# ==========================================
# 6. Padding (identical convention to training: zero-pad to MAX_FRAMES)
# ==========================================
_dummy_frame = [0.0] * 32
FEATURE_DIM = len(build_feature_vector(_dummy_frame, _dummy_frame, _dummy_frame))
print(f"Per-frame feature vector dimensionality: {FEATURE_DIM}")


def pad_sequence_inference(seq, max_frames=MAX_FRAMES):
    seq = np.array(seq)
    if len(seq) == 0:
        return np.zeros((max_frames, FEATURE_DIM))
    if len(seq) < max_frames:
        padding = np.zeros((max_frames - len(seq), seq.shape[1]))
        seq = np.vstack([seq, padding])
    return seq[:max_frames]


# ==========================================
# 7. Skeleton video rendering (black canvas, core joints only,
#    prediction banner on top)
# ==========================================
def save_skeleton_video(pixel_frames, width, height, fps, output_path, predicted_label, confidence):
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(output_path, fourcc, fps, (width, height))

    overlay_text = f"{predicted_label} ({confidence * 100:.1f}%)"
    accent_color = (0, 140, 255) if predicted_label != "NoPunch" else (100, 100, 100)

    for coords in pixel_frames:
        canvas = np.zeros((height, width, 3), dtype=np.uint8)

        for a, b in BONES:
            pt1 = tuple(coords[a].astype(int))
            pt2 = tuple(coords[b].astype(int))
            cv2.line(canvas, pt1, pt2, (255, 255, 255), 2)

        for pt in coords:
            cv2.circle(canvas, tuple(pt.astype(int)), 4, (0, 255, 0), -1)

        cv2.rectangle(canvas, (0, 0), (width, 60), (0, 0, 0), -1)
        cv2.rectangle(canvas, (10, 10), (25, 50), accent_color, -1)
        cv2.putText(canvas, f"Prediction: {overlay_text}", (35, 40),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.9, (255, 255, 255), 2, cv2.LINE_AA)

        out.write(canvas)

    out.release()


# ==========================================
# 8. Per-clip pipeline
# ==========================================
def process_video_clip(video_path, pose_detector):
    feature_frames, pixel_frames, width, height, fps = extract_raw_sequence(video_path, pose_detector)
    if len(feature_frames) == 0:
        return None

    normalized = normalize_sequence(feature_frames)
    feature_sequence = process_punch_sequence(normalized)

    return {
        "feature_sequence": feature_sequence,
        "pixel_frames": pixel_frames,
        "width": width,
        "height": height,
        "fps": fps,
    }


# ==========================================
# 9. Main
# ==========================================
def main():
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    print(f"Loading LSTM Model from: {MODEL_PATH}")
    model = tf.keras.models.load_model(MODEL_PATH)
    print("Model loaded successfully.")

    model_num_classes = model.output_shape[-1]
    if model_num_classes != len(CLASS_NAMES):
        raise ValueError(
            f"Model outputs {model_num_classes} classes but CLASS_NAMES has "
            f"{len(CLASS_NAMES)} entries ({CLASS_NAMES}). Update the ALL_CLASSES "
            f"enabled flags in the CONFIG block to match the classes this "
            f"specific model was trained on before running inference."
        )
    print(f"Class count verified: model and CLASS_NAMES both have {len(CLASS_NAMES)} classes.")
    print(f"Classes: {CLASS_NAMES}")
    print(f"Skeleton outputs will be saved to: {OUTPUT_DIR}\n")

    mp_pose = mp.solutions.pose
    pose = mp_pose.Pose(static_image_mode=False)  # same as training extraction; all other args left at MediaPipe defaults

    if os.path.isfile(INPUT_DIR):
        video_files = [INPUT_DIR]
    elif os.path.isdir(INPUT_DIR):
        video_files = [
            os.path.join(INPUT_DIR, f) for f in os.listdir(INPUT_DIR)
            if f.lower().endswith(('.mp4', '.avi', '.mov', '.mkv'))
        ]
    else:
        raise FileNotFoundError(f"Input path does not exist: {INPUT_DIR}")

    print(f"Found {len(video_files)} video file(s) to process.\n")
    print("=" * 75)

    for vid_path in video_files:
        filename = os.path.basename(vid_path)
        base_name, ext = os.path.splitext(filename)

        result = process_video_clip(vid_path, pose)
        if result is None:
            print(f"[{filename}] - SKIPPED (No valid pose tracked)")
            continue

        padded_seq = pad_sequence_inference(result["feature_sequence"], MAX_FRAMES)
        input_tensor = np.expand_dims(padded_seq, axis=0)

        probabilities = model.predict(input_tensor, verbose=0)[0]
        pred_idx = int(np.argmax(probabilities))
        confidence = probabilities[pred_idx]
        predicted_label = CLASS_NAMES[pred_idx] if confidence > CONFIDENCE_THRESHOLD else "NoPunch"

        top_n = min(3, len(CLASS_NAMES))
        top_indices = np.argsort(probabilities)[::-1][:top_n]
        top_str = ", ".join([f"{CLASS_NAMES[i]}: {probabilities[i] * 100:.1f}%" for i in top_indices])

        out_filename = f"{base_name}_{predicted_label.replace(' ', '_')}_skeleton{ext}"
        out_filepath = os.path.join(OUTPUT_DIR, out_filename)
        save_skeleton_video(
            result["pixel_frames"], result["width"], result["height"], result["fps"],
            out_filepath, predicted_label, confidence
        )

        print(f"Clip:                 {filename}")
        print(f"Processed Frames:     {len(result['feature_sequence'])}")
        print(f"Predicted Label:      {predicted_label}")
        print(f"Confidence:           {confidence * 100:.2f}%")
        print(f"Top-{top_n} Probabilities:  [{top_str}]")
        print(f"Saved Skeleton To:    {out_filepath}")
        print("-" * 75)

    pose.close()


if __name__ == "__main__":
    main()